# 01 — Load & Filter NIST Isotherms

**Purpose:** Load the NIST Isotherm Database (ISODB), apply successive quality filters to isolate experimental methane adsorption isotherms on pristine MOFs, normalise units to mmol/g and bar, handle duplicates, apply coherence corrections, and export the cleaned dataset.

**Inputs:**
- `isodb-library/` — local git clone of the NIST ISODB  

**Outputs:**
- `data/nistdb.pickle` — cached cleaned ISODB (with commit tracking)
- `data/mof_adsorbents_for_cif_matching.json` — filtered MOF adsorbents with DOIs
- `data/filtered_isotherms.xlsx` — formatted Excel export of filtered MOFs
- `data/removed_isotherms_all_filters.xlsx` — log of removed entries by filter stage
- `data/mof_adsorbent_names.txt` — plain text list of retained MOF names

## 1. Load ISODB and Clean Raw Data

Reads all isotherm JSON files from the local ISODB git repo. Uses a pickle cache keyed to the current git commit so subsequent runs load in ~1 s instead of ~14 s. After loading, two data-cleaning fixes are applied in-place:

1. **`total_adsorption` backfill** — older multi-component records leave this field `null`; we sum the per-species values.
2. **`category` correction** — some records have a blank category; when every isotherm in the same paper shares one category (exp/sim), we propagate it.

In [ ]:
import git, json, pickle, time, os
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import Counter, defaultdict

isodb_git_path = Path("isodb-library")
pickle_path    = Path("data/nistdb.pickle")
commit_file    = Path("data/.nistdb_commit")

current_commit = git.Repo(isodb_git_path).head.object.hexsha
print("Last commit:", current_commit[:8])

# ── Load (pickle cache or fresh JSON scan) ──────────────────────────────────
cache_valid = (
    pickle_path.exists()
    and commit_file.exists()
    and commit_file.read_text().strip() == current_commit
)

if cache_valid:
    t0 = time.perf_counter()
    with open(pickle_path, "rb") as f:
        nistdb = pickle.load(f)
    print(f"Loaded from pickle in {time.perf_counter()-t0:.1f}s")
else:
    t0 = time.perf_counter()
    nistdb = {'Adsorbents': [], 'Adsorbates': [], 'Bibliography': [], 'Isotherms': []}
    for key in nistdb:
        if key == 'Isotherms':
            for p in tqdm((isodb_git_path / "Library").glob("10*/*"), desc="Isotherms"):
                with open(p, "r", encoding="utf-8") as f:
                    nistdb[key].append(json.load(f))
        else:
            for p in (isodb_git_path / "Library").glob(f"{key}/*"):
                with open(p, "r", encoding="utf-8") as f:
                    nistdb[key].append(json.load(f))
    print(f"Loaded from JSON in {time.perf_counter()-t0:.1f}s")

for key in nistdb:
    print(f"  {key}: {len(nistdb[key])}")

In [ ]:
# ── Fix 1: backfill total_adsorption ─────────────────────────────────────────
isos = nistdb['Isotherms']
fixes = 0
for iso in isos:
    if iso['isotherm_data'][0]['total_adsorption'] is None:
        fixes += 1
        for pt in iso['isotherm_data']:
            pt['total_adsorption'] = float(np.sum([s['adsorption'] for s in pt['species_data']]))
print(f"total_adsorption backfilled: {fixes}/{len(isos)} isotherms")

# ── Fix 2: correct blank category from bibliography ─────────────────────────
doi_to_cat = {b['DOI']: b['categories'] for b in nistdb['Bibliography']}
cat_fixes, cat_known, cat_unknown = 0, 0, 0
for iso in isos:
    if iso['category'] == "":
        if iso['DOI'] in doi_to_cat and len(doi_to_cat[iso['DOI']]) == 1:
            iso['category'] = doi_to_cat[iso['DOI']][0]
            cat_fixes += 1
        else:
            cat_unknown += 1
    else:
        cat_known += 1
print(f"category corrected: {cat_fixes} fixed, {cat_known} known, {cat_unknown} unresolved")

# ── Save updated pickle ─────────────────────────────────────────────────────
pickle_path.parent.mkdir(parents=True, exist_ok=True)
with open(pickle_path, "wb") as f:
    pickle.dump(nistdb, f)
commit_file.write_text(current_commit)
print(f"Pickle saved for commit {current_commit[:8]}")

## 2. Sequential Isotherm Filters

Four successive filters narrow the dataset to experimental, single-component methane isotherms measured at near-ambient temperatures with mass-based adsorption units.

In [ ]:
isos = nistdb['Isotherms']
print(f"Starting isotherms: {len(isos)}")

# Filter 1: experimental only
filtered = [i for i in isos if i['category'] == 'exp']
print(f"After experimental filter: {len(filtered)}")

# Filter 2: single-component methane
filtered = [i for i in filtered
            if len(i['adsorbates']) == 1 and i['adsorbates'][0]['name'] == 'Methane']
print(f"After methane filter: {len(filtered)}")

# Filter 3: temperature range 273–323 K
temp_min, temp_max = 273, 323
filtered = [i for i in filtered if temp_min <= i['temperature'] <= temp_max]
print(f"After temperature filter ({temp_min}–{temp_max} K): {len(filtered)}")

# Filter 4: remove volumetric denominators (mol, cm3, m3, unit cell, etc.)
bad_denoms = {'mol', 'volume', 'cm3', 'm3', 'cm2', 'm2', 'unit cell', 'unitcell', 'formula'}
filtered = [i for i in filtered
            if not ("/" in i['adsorptionUnits'] and i['adsorptionUnits'].split("/")[-1] in bad_denoms)]
print(f"After bad-denominator filter: {len(filtered)}")

## 3. Unit Normalisation to mmol/g

Converts all remaining adsorption units to **mmol/g** using molar mass of methane (16.043 g/mol) and STP molar volume (22,414 cm³/mol). Isotherms with unrecognised units are removed and logged.

In [ ]:
M_CH4 = 16.043      # g/mol
V_STP = 22_414.0    # cm³/mol at STP

def _conversion_factor(unit):
    """Multiplier to convert adsorption value to mmol/g. None if unrecognised."""
    u = unit.strip().lower()
    u_ns = u.replace(" ", "")
    lookup = {
        'mmol/g': 1.0, 'mol/kg': 1.0, 'mmol/kg': 1e-3,
        'mg/g': 1.0/M_CH4, 'g/g': 1000.0/M_CH4, 'g/l': 1.0/M_CH4, 'g/ml': 1000.0/M_CH4,
    }
    if u in lookup: return lookup[u]
    if u in ('µmol/g','umol/g','μmol /g') or u_ns in ('µmol/g','umol/g','μmol/g'): return 1e-3
    if u in ('cm3(stp)/g','cc(stp)/g','ml(stp)/g','cm3 (stp)/g','cc (stp)/g','ml (stp)/g','ml/g'):
        return 1000.0/V_STP
    if u in ('l(stp)/g','l (stp)/g'): return 1e6/V_STP
    if u in ('wt%','wt. %','wt.%','weight %','uptake%','uptake %'): return 10.0/M_CH4
    return None

# Audit current units
print(f"{'Unit':<30} {'Count':>6}")
print("-"*40)
for u, c in sorted(Counter(i['adsorptionUnits'] for i in filtered).items(), key=lambda x:-x[1]):
    tag = "" if u == 'mmol/g' else " ← convert"
    print(f"  {u:<28} {c:>6}{tag}")

# Convert in-place
unknown_units, removed_unknown = set(), []
for iso in filtered:
    f = _conversion_factor(iso['adsorptionUnits'])
    if f is None:
        unknown_units.add(iso['adsorptionUnits'])
        removed_unknown.append(iso)
        continue
    if f != 1.0:
        for pt in iso['isotherm_data']:
            if pt.get('total_adsorption') is not None:
                pt['total_adsorption'] *= f
            for sp in pt.get('species_data', []):
                if sp.get('adsorption') is not None:
                    sp['adsorption'] *= f
        iso['adsorptionUnits'] = 'mmol/g'

filtered = [i for i in filtered if i not in removed_unknown]
print(f"\nConverted to mmol/g. Removed {len(removed_unknown)} with unknown units: {unknown_units}")
print(f"Remaining: {len(filtered)}")

## 4. Non-MOF Removal

Three-stage filter to remove non-MOF materials that passed the basic criteria filters.

**Stage 1 — Keyword filter:** removes entries containing non-MOF keywords (zeolite, carbon, graphene, etc.) unless the name also matches a known MOF family pattern.

**Stage 2 — Metal-centre filter:** MOFs require metal nodes. Removes entries whose name contains no recognisable metal symbol and matches no known MOF family prefix.

**Stage 3 — Composite exclusion:** removes composites (containing @, %, nano, etc.) and specific entries identified during manual inspection.

In [ ]:
import re

# ── Global whitelist: confirmed MOFs that would be incorrectly removed ───────
_mof_whitelist = {
    '[La(BTB)(H2O)3DMF]n Activated',
    '(Me2NH2)2(DMF)9(H2O)5',
    '2,4,6-tris-(4-carboxyphenoxy)-1,3,5-triazine (H3tcpt)',
    'Sc2(O2CC6H4CO2)3',
}

# ── Build DOI lookup for all isotherms ───────────────────────────────────────
_name_to_dois = defaultdict(set)
for iso in isos:
    _name_to_dois[iso['adsorbent']['name']].add(iso['DOI'])

# ── Build synonym lookup ─────────────────────────────────────────────────────
name_to_synonyms = {}
for ads in nistdb['Adsorbents']:
    name_to_synonyms[ads['name']] = ads.get('synonyms', [])

In [ ]:
# ── STAGE 1: Keyword filter ───────────────────────────────────────────────────
non_mof_keywords = ['zeolite', 'carbon', 'graphene', 'graphite', 'silica', 'alumina']
_boundary_keywords = {
    'activated': re.compile(r'\bactivated\b', re.IGNORECASE),
    'cof':       re.compile(r'\bcof[-\s]?\d*\b', re.IGNORECASE),
}
_mof_rescue_patterns = [
    'mil-', 'mof-', 'mof(', 'hkust', 'irmof', 'zif-', 'uio', '-mof',
    'cu-btc', 'cubtc', 'btc', 'pcn-', 'nott-', 'bio-mof',
]
_bracket_formula_mof = re.compile(r'^activated\s+[\[\{]|[\]\}]\w*\s+activated', re.IGNORECASE)

def _is_mof_rescue(name_lower):
    return any(p in name_lower for p in _mof_rescue_patterns) or bool(_bracket_formula_mof.match(name_lower))

def not_mof(name, synonyms):
    for n in [name.lower()] + [s.lower() for s in synonyms]:
        if any(kw in n for kw in non_mof_keywords):
            if not _is_mof_rescue(n): return True
        for kw, pat in _boundary_keywords.items():
            if pat.search(n) and not _is_mof_rescue(n): return True
    return False

before_kw = len(filtered)
filtered = [i for i in filtered
            if i['adsorbent']['name'] in _mof_whitelist
            or not not_mof(i['adsorbent']['name'], name_to_synonyms.get(i['adsorbent']['name'], []))]
removed_keyword = before_kw - len(filtered)
print(f"Stage 1 (keyword): removed {removed_keyword}, remaining {len(filtered)}")

In [ ]:
# ── STAGE 2: Metal-centre / MOF-family filter ───────────────────────────────
metals = [
    'Mg','Ca','Sr','Ba','Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn',
    'Zr','Nb','Mo','Ru','Rh','Pd','Ag','Cd','Hf','W','Re','Ir','Pt',
    'Al','Ga','In','Sn','Pb','Bi',
    'La','Ce','Pr','Nd','Sm','Eu','Gd','Tb','Dy','Ho','Er','Tm','Yb','Lu','Y',
]

known_mof_families = [
    'hkust','irmof','mof-','mof(','bio-mof','soc-mof','mil-','zif-','uio-','uio(',
    'pcn-','nott-','mof-5','mof-177','mof-74','mof-399','mof-505','mof-210','-mof',
    'cubtc','cu-btc','cu3(btc)','nu-','utsa','snu-','dut-','umcm','cau-','cuk-',
    'fji-','fir-','comoc','cpm-','cpf-','sdu-','zju-','zjnu','znju','nju-','upc-',
    'uhm-','maf-','elm-','rod-','sion-','lifm','jlu-','ipm-','sust-','but-',
    'tmof','moaaf','peconf','calf-','aemof','pcu-','inoh-','juc-','hit-','snnu-',
    'gdmof','ctgu-','smes-','kmof-','cau(','hmof','mmof','nmof','smof','tmof',
    'dmof','emof','fmof','amof','bmof','cmof','jmof','zmof','ymof','pmof','omof','lmof',
    'bmmof','nkmof','njumof','kgmmof','gdmof','npmof','nupf',
    'fjtmof','zjumof','zrmof','znmof','cumof','femof','scmof','mgmof',
    'nimof','comof','mnmof','crmof','vmof','timof','cdmof','agmof',
    'pdmof','irmof','lumof','srmof','bamof','pbmof',
    'piza','stam','slug','sifsix','crofour',
]

_metal_pattern = re.compile(
    r'(?<![A-Za-z])(' + '|'.join(re.escape(m) for m in metals) + r')(?=[^a-z]|$)'
)

def has_metal_or_family(name, synonyms):
    for n in [name] + synonyms:
        nl = n.lower()
        if any(f in nl for f in known_mof_families): return True
        if _metal_pattern.search(n): return True
    return False

before_metal = len(filtered)
filtered = [i for i in filtered
            if i['adsorbent']['name'] in _mof_whitelist
            or has_metal_or_family(i['adsorbent']['name'], name_to_synonyms.get(i['adsorbent']['name'], []))]
removed_metal = before_metal - len(filtered)
print(f"Stage 2 (metal/family): removed {removed_metal}, remaining {len(filtered)}")

In [ ]:
# ── STAGE 3: Composite / non-MOF exclusion ───────────────────────────────────
known_mof_families = [f for f in known_mof_families if f != 'peconf']

_exclude_substrings = [
    '%', '@', 'coal', 'graphite oxide', 'peconf', 'zncl2', 'tio2', 'alpo',
    'clinoptilolite', 'stress', 'microwave', 'powder', 'powdered', 'nano', 'shaped', 'sol.ht',
]
_exclude_regex = [
    re.compile(r'[-/\s]GO\b'),
    re.compile(r'^bio-', re.IGNORECASE),
    re.compile(r'\bAC\b'),
    re.compile(r'\bPMO\d*\b'),
    re.compile(r'(?<![A-Za-z])CO2(?![(\d])', re.IGNORECASE),
]
_exclude_exact = {'Cu(I)Y', 'Cu-MOF', 'MIL-53(Al)'}

def is_excluded_composite(name):
    if name in _exclude_exact: return True
    nl = name.lower()
    if any(p in nl for p in _exclude_substrings): return True
    if any(p.search(name) for p in _exclude_regex): return True
    return False

before_comp = len(filtered)
filtered = [i for i in filtered
            if i['adsorbent']['name'] in _mof_whitelist or not is_excluded_composite(i['adsorbent']['name'])]
removed_composite = before_comp - len(filtered)
print(f"Stage 3 (composite): removed {removed_composite}, remaining {len(filtered)}")

## 5. Duplicate Handling

Groups isotherms by (hashkey, temperature, adsorbate) and resolves duplicates using a curated keep-list (`data/duplicate_keep_filenames.txt`). When a group has a file on the keep-list, only that file is retained; otherwise the first isotherm in the group is kept.

In [ ]:
def _adsorbate_sig(iso):
    return tuple(sorted(a.get("name","") for a in iso.get("adsorbates",[])))

def _dup_key(iso):
    t = iso.get("temperature")
    if t is not None: t = round(float(t), 3)
    return (iso["adsorbent"]["hashkey"], t, _adsorbate_sig(iso))

def _norm_fn(name):
    s = str(name).strip()
    if s.lower().endswith(".json"): s = s[:-5]
    return s.lower()

def _load_fn_list(path):
    out = set()
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.split("#",1)[0].strip()
        if not line: continue
        parts = [p.strip() for p in line.split(",") if p.strip()]
        out.add(_norm_fn(parts[-1] if parts else line))
    return out

# Group duplicates
dup_groups = defaultdict(list)
for iso in filtered:
    dup_groups[_dup_key(iso)].append(iso)

multi = {k:v for k,v in dup_groups.items() if len(v)>1}
print(f"Duplicate groups (same hashkey+T+adsorbate): {len(multi)}")

# Resolve using keep-list
keep_fns = _load_fn_list(Path("data/duplicate_keep_filenames.txt"))
resolved = []
for key, group in dup_groups.items():
    if len(group) == 1:
        resolved.append(group[0])
    else:
        kept = [i for i in group if _norm_fn(i.get("filename","")) in keep_fns]
        resolved.append(kept[0] if kept else group[0])

print(f"After deduplication: {len(resolved)} (was {len(filtered)})")
filtered = resolved

## 6. Coherence Corrections

Filename-specific fixes for isotherms where units are internally consistent but numerically wrong (identified during visual audit). Three correction types:

1. **PSI → bar:** pressure values are in PSI, multiply by 100000/14.5038
2. **cm³ → mmol:** adsorption values are in cm³(STP)/g not mmol/g, multiply by 1000/22414
3. **Pressure /100:** pressure is off by factor of 100

In [ ]:
# ── Remove isotherms flagged during visual audit ────────────────────────────
audit_remove = _load_fn_list(Path("data/audit_removal_filenames.txt"))
before_audit = len(filtered)
filtered = [i for i in filtered if _norm_fn(i.get("filename","")) not in audit_remove]
print(f"Removed from audit list: {before_audit - len(filtered)}, remaining: {len(filtered)}")

# ── PSI → bar correction ────────────────────────────────────────────────────
psi_files = {_norm_fn(x) for x in [
    "10.1021jp304631m.Isotherm15", "10.1021jp304631m.Isotherm17",
    "10.1021jp304631m.Isotherm30", "10.1021jp304631m.Isotherm31",
    "10.1021jp304631m.Isotherm39", "10.1021jp304631m.Isotherm40",
]}
psi_factor = 100000.0 / 14.5038
psi_count = 0
for iso in filtered:
    if _norm_fn(iso.get("filename","")) in psi_files:
        max_p = max((pt.get("pressure",0) or 0) for pt in iso['isotherm_data'])
        if max_p < 0.02:  # guard against double-application
            for pt in iso['isotherm_data']:
                if pt.get("pressure") is not None: pt["pressure"] *= psi_factor
            psi_count += 1
print(f"PSI→bar corrections: {psi_count}")

# ── cm³→mmol correction ─────────────────────────────────────────────────────
cm3_files = {_norm_fn(x) for x in [
    "10.1039C3ta11548h.isotherm17", "10.1039C3ta11548h.isotherm18",
    "10.1039C3ta11548h.isotherm15", "10.1039C3ta11548h.isotherm16",
    "10.1039C3ta11840a.Isotherm6",
]}
cm3_factor = 1000.0 / 22414.0
cm3_count = 0
for iso in filtered:
    if _norm_fn(iso.get("filename","")) in cm3_files:
        max_q = max((pt.get("total_adsorption",0) or 0) for pt in iso['isotherm_data'])
        if max_q > 2.0:  # guard
            for pt in iso['isotherm_data']:
                if pt.get("total_adsorption") is not None: pt["total_adsorption"] *= cm3_factor
                for sp in pt.get("species_data",[]): 
                    if sp.get("adsorption") is not None: sp["adsorption"] *= cm3_factor
            cm3_count += 1
print(f"cm³→mmol corrections: {cm3_count}")

# ── Pressure /100 correction ────────────────────────────────────────────────
div100_files = {_norm_fn(x) for x in [
    "10.1039C3cc48275h.Isotherm10", "10.1039C3cc48275h.Isotherm11",
    "10.1039C3cc48275h.Isotherm12", "10.1039C3cc48275h.Isotherm13",
    "10.1039C3cc48275h.Isotherm14", "10.1016j.micromeso.2011.09.006.isotherm5",
]}
div100_count = 0
for iso in filtered:
    if _norm_fn(iso.get("filename","")) in div100_files:
        max_p = max((pt.get("pressure",0) or 0) for pt in iso['isotherm_data'])
        if max_p > 40.0:  # guard
            for pt in iso['isotherm_data']:
                if pt.get("pressure") is not None: pt["pressure"] /= 100.0
            div100_count += 1
print(f"Pressure /100 corrections: {div100_count}")

## 7. Visual Audit Export (Optional)

Exports a pairgrid PNG of all remaining unique isotherms for visual inspection. Set `EXPORT_PAIRGRID = False` to skip.

In [ ]:
from math import ceil
import matplotlib.pyplot as plt

EXPORT_PAIRGRID = True
GRID_COLS, GRID_ROWS, DPI = 11, 12, 300

if EXPORT_PAIRGRID:
    unique_isos = sorted(filtered, key=lambda i: (i['adsorbent']['name'], i.get('temperature',0), i.get('filename','')))
    n = len(unique_isos)
    per_page = GRID_COLS * GRID_ROWS
    pages = ceil(n / per_page)

    for page in range(pages):
        batch = unique_isos[page*per_page : (page+1)*per_page]
        rows = ceil(len(batch) / GRID_COLS)
        fig, axes = plt.subplots(rows, GRID_COLS, figsize=(GRID_COLS*3.6, rows*2.8), squeeze=False, constrained_layout=True)
        flat = axes.reshape(-1)
        for ax, iso in zip(flat, batch):
            pts = [p for p in iso.get('isotherm_data',[]) if p.get('pressure') and p.get('total_adsorption')]
            if pts:
                ax.plot([p['pressure'] for p in pts], [p['total_adsorption'] for p in pts],
                        marker='o', lw=0.9, ms=2, alpha=0.85)
            ax.set_title(f"{iso['adsorbent']['name']}\nT={iso.get('temperature','?')}K", fontsize=7)
            ax.tick_params(labelsize=6); ax.grid(True, alpha=0.2)
        for ax in flat[len(batch):]: ax.axis('off')
        fig.suptitle(f"Unique isotherms (page {page+1}/{pages}, n={n})", fontsize=12)
        fig.savefig(f"data/unique_isotherms_pairgrid_p{page+1}.png", dpi=DPI)
        plt.close(fig)
    print(f"Exported {pages} pairgrid page(s) to data/")
else:
    print("Pairgrid export skipped (EXPORT_PAIRGRID=False)")

## 8. Export Filtered Dataset

Saves the final filtered MOF adsorbent list as JSON (for downstream CIF matching) and as a formatted Excel workbook. Also exports a removal log and a plain-text name list.

In [ ]:
# ── Collect unique adsorbents ─────────────────────────────────────────────────
unique_hashkeys = set(i['adsorbent']['hashkey'] for i in filtered)
hashkey_to_temps = defaultdict(set)
for iso in filtered:
    hashkey_to_temps[iso['adsorbent']['hashkey']].add(iso['temperature'])

output_records = []
for ads in nistdb['Adsorbents']:
    if ads['hashkey'] in unique_hashkeys:
        all_dois = sorted(_name_to_dois.get(ads['name'], {'N/A'}))
        temps = sorted(hashkey_to_temps.get(ads['hashkey'], set()))
        output_records.append({
            'name': ads['name'], 'hashkey': ads['hashkey'],
            'DOI': all_dois[0], 'DOIs': all_dois,
            'Temperatures': temps, 'synonyms': ads['synonyms'],
        })

print(f"Unique MOF adsorbents: {len(output_records)}")

# ── JSON export ──────────────────────────────────────────────────────────────
with open('data/mof_adsorbents_for_cif_matching.json', 'w', encoding='utf-8') as f:
    json.dump(output_records, f, indent=2, ensure_ascii=False)
print("Saved: data/mof_adsorbents_for_cif_matching.json")

# ── Name list export ─────────────────────────────────────────────────────────
with open('data/mof_adsorbent_names.txt', 'w', encoding='utf-8') as f:
    for rec in sorted(output_records, key=lambda r: r['name']):
        f.write(rec['name'] + '\n')
print("Saved: data/mof_adsorbent_names.txt")

In [ ]:
# ── Excel export (formatted) ──────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook(); ws = wb.active; ws.title = "Filtered MOF Adsorbents"
hdr_font = Font(name="Arial", bold=True, size=11, color="FFFFFF")
hdr_fill = PatternFill("solid", fgColor="2F5496")
hdr_align = Alignment(horizontal="center", vertical="center")
thin = Side(style="thin", color="D9D9D9")
border = Border(left=thin, right=thin, top=thin, bottom=thin)
body_font = Font(name="Arial", size=10)
alt_fill = PatternFill("solid", fgColor="F2F2F2")
link_font = Font(name="Arial", size=10, color="0563C1", underline="single")

headers = ["#", "Adsorbent Name", "Hashkey", "DOI", "DOI Link"]
widths  = [5, 45, 38, 30, 50]
for ci, (h,w) in enumerate(zip(headers, widths), 1):
    c = ws.cell(row=1, column=ci, value=h)
    c.font, c.fill, c.alignment, c.border = hdr_font, hdr_fill, hdr_align, border
    ws.column_dimensions[c.column_letter].width = w

for i, rec in enumerate(sorted(output_records, key=lambda r: r['name']), 1):
    doi = sorted(_name_to_dois.get(rec['name'], {"N/A"}))[0]
    row = i + 1
    ws.cell(row=row, column=1, value=i).font = body_font
    ws.cell(row=row, column=2, value=rec['name']).font = body_font
    ws.cell(row=row, column=3, value=rec['hashkey']).font = body_font
    ws.cell(row=row, column=4, value=doi).font = body_font
    lc = ws.cell(row=row, column=5)
    if doi != "N/A":
        url = f"https://doi.org/{doi}"
        lc.value, lc.hyperlink, lc.font = url, url, link_font
    for ci in range(1, 6):
        ws.cell(row=row, column=ci).border = border
        if row % 2 == 0: ws.cell(row=row, column=ci).fill = alt_fill

ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = "A2"
wb.save("data/filtered_isotherms.xlsx")
print("Saved: data/filtered_isotherms.xlsx")

print(f"\n{'='*50}")
print(f"FINAL SUMMARY")
print(f"{'='*50}")
print(f"Isotherms: {len(filtered)}")
print(f"Unique adsorbents: {len(output_records)}")